# Oasis Infobyte Data Analytics Internship
## Project 1: Sentiment Analysis System
**Author:** Jasmin Jamadar  
**Role:** Data Analyst Intern  
**Project Objective:** Build a Natural Language Processing (NLP) system to classify text reviews into Positive, Neutral, or Negative categories and extract insights about public sentiment.

---



### Project Outline
1. **Environment Setup & Library Imports** - Loading essential Python and Data Science libraries.
2. **Dataset Loading & Initial Inspection** - Inspecting the shape, columns, and sample records.
3. **Data Preprocessing & Cleaning** - Lowercasing, removing URLs, punctuation, special characters, tokenization, and stopwords.
4. **Exploratory Data Analysis (EDA)** - Analyzing class distributions and text characteristics.
5. **Data Visualization** - Generating sentiment distribution plots and positive/negative word clouds.
6. **Feature Engineering (TF-IDF Vectorization)** - Converting raw text to numerical representation.
7. **Model Training & Evaluation** - Building, training, and testing:
   - Logistic Regression
   - Multinomial Naive Bayes
8. **Model Comparison & Confusion Matrices** - Evaluating and comparing metrics (Accuracy, Precision, Recall, F1-Score).
9. **Key Findings & Internship Conclusion** - Summarizing model performance and analytical insights.



### 1. Environment Setup & Library Imports
We will import standard NLP, data modeling, and visualization libraries. Make sure you have libraries like `nltk`, `wordcloud`, and `scikit-learn` installed.



In [ ]:
import os
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
stop_words = set(stopwords.words('english'))
print("Setup complete. Libraries imported and NLTK resources downloaded.")


### 2. Dataset Loading & Initial Inspection
We load the dataset `Twitter_Data.csv` from the `Dataset` directory and inspect its dimensions, headers, and missing values.



In [ ]:
# Path to the dataset
dataset_path = os.path.join("..", "Dataset", "Twitter_Data.csv")

# Load dataset
df = pd.read_csv(dataset_path)
print(f"Dataset Loaded Successfully! Shape: {df.shape}")
print("\n--- First 5 Rows ---")
print(df.head())
print("\n--- Dataset Information ---")
df.info()


### 3. Data Preprocessing & Cleaning
Text cleaning is the most critical phase in NLP. We will perform:
- **Missing Value Handling**: Drop rows containing null texts or labels.
- **Conversion to Lowercase**: Keep vocabulary size consistent.
- **URL & HTML Removal**: Remove link paths and tags that don't contribute to sentiment.
- **Special Characters Removal**: Keep only alphabetic characters and spaces.
- **Stopwords Removal**: Remove common English words (like 'the', 'is', 'at') that do not carry sentimental value.
- **Tokenization**: Segment words into lists of tokens.



In [ ]:
# Check for missing values
print("Null counts before cleaning:")
print(df.isnull().sum())

# Drop missing values
df = df.dropna().reset_index(drop=True)
df['category'] = df['category'].astype(int)
print(f"Shape after dropping nulls: {df.shape}")

# Preprocessing function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters, numbers and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenization & Stopwords removal
    tokens = text.split()
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(cleaned_tokens)

# Run preprocessing
print("\nPreprocessing reviews (this may take a few seconds)...")
t0 = time.time()
df['processed_text'] = df['clean_text'].apply(clean_text)
print(f"Preprocessing completed in {time.time() - t0:.2f} seconds.")

# Remove any rows that became empty after cleaning
df = df[df['processed_text'] != ""].reset_index(drop=True)
print(f"Final preprocessed dataset shape: {df.shape}")


In [ ]:
# Verify cleaning results with a sample
idx = 10
print("--- Raw Text ---")
print(df['clean_text'].iloc[idx])
print("\n--- Cleaned Text ---")
print(df['processed_text'].iloc[idx])
print(f"\nCategory Label: {df['category'].iloc[idx]}")


### 4. Exploratory Data Analysis & Sentiment Distribution
We visualize the count and percentage distribution of our target classes:
- **-1**: Negative
- **0**: Neutral
- **1**: Positive



In [ ]:
# Check distribution
label_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
df['sentiment_label'] = df['category'].map(label_map)
counts = df['sentiment_label'].value_counts()

# Set up the plotting canvas
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Bar Plot
sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=axes[0], hue=counts.index, legend=False)
axes[0].set_title('Distribution of Sentiments (Counts)', fontsize=15, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Number of Tweets')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha='center', fontweight='bold')

# 2. Pie Chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=140,
            colors=['#4f46e5', '#10b981', '#f59e0b'], textprops={'fontsize': 12, 'fontweight': 'bold'},
            explode=(0.02, 0.02, 0.02))
axes[1].set_title('Distribution of Sentiments (%)', fontsize=15, fontweight='bold')

plt.tight_layout()
# Save chart to Visualizations folder
os.makedirs(os.path.join("..", "Visualizations"), exist_ok=True)
plt.savefig(os.path.join("..", "Visualizations", "sentiment_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()


### 5. Word Clouds Visualizations
To understand what words are key drivers of sentiments, we generate Word Clouds for the **Positive** and **Negative** subsets of reviews.



In [ ]:
# Extract text for Positive and Negative sentiments
positive_text = " ".join(df[df['category'] == 1]['processed_text'])
negative_text = " ".join(df[df['category'] == -1]['processed_text'])

# Word Cloud for Positive Reviews
print("Generating Positive Word Cloud...")
wc_pos = WordCloud(width=800, height=450, background_color='white', colormap='summer',
                   max_words=150, random_state=42).generate(positive_text)
plt.figure(figsize=(10, 6))
plt.imshow(wc_pos, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Positive Sentiments', fontsize=15, fontweight='bold', pad=15)
plt.savefig(os.path.join("..", "Visualizations", "wordcloud_positive.png"), dpi=150, bbox_inches='tight')
plt.show()

# Word Cloud for Negative Reviews
print("Generating Negative Word Cloud...")
wc_neg = WordCloud(width=800, height=450, background_color='white', colormap='autumn',
                   max_words=150, random_state=42).generate(negative_text)
plt.figure(figsize=(10, 6))
plt.imshow(wc_neg, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Negative Sentiments', fontsize=15, fontweight='bold', pad=15)
plt.savefig(os.path.join("..", "Visualizations", "wordcloud_negative.png"), dpi=150, bbox_inches='tight')
plt.show()


### 6. TF-IDF Feature Extraction
Text data must be vectorized before classification. We apply:
- **Train/Test Split**: 80% Training, 20% Testing (stratified on sentiment).
- **TF-IDF Vectorization**: Extract Term Frequency-Inverse Document Frequency features with `max_features=25000` and including both unigrams and bigrams (`ngram_range=(1,2)`).



In [ ]:
# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], 
    df['category'], 
    test_size=0.20, 
    random_state=42, 
    stratify=df['category']
)

print(f"Training split: {X_train.shape[0]} samples")
print(f"Testing split: {X_test.shape[0]} samples")

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")


### 7. Model Training & Evaluation
We train and evaluate two popular classification algorithms:
1. **Logistic Regression** (solver='lbfgs' for multiclass support)
2. **Multinomial Naive Bayes** (fast, probabilistic baseline for text)



In [ ]:
results = {}
confusion_matrices = {}

# --- 1. Logistic Regression ---
print("Training Logistic Regression Model...")
t_start = time.time()
lr_model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
t_train_lr = time.time() - t_start

# Evaluation
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)
lr_precision, lr_recall, lr_f1, _ = precision_recall_fscore_support(y_test, lr_preds, average='macro')

results['Logistic Regression'] = {
    'Accuracy': lr_acc,
    'Precision': lr_precision,
    'Recall': lr_recall,
    'F1-Score': lr_f1,
    'Train Time (s)': t_train_lr
}
confusion_matrices['Logistic Regression'] = confusion_matrix(y_test, lr_preds)

print(f"Logistic Regression trained in {t_train_lr:.2f} seconds.")
print("\nClassification Report:")
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Neutral', 'Positive']))


In [ ]:
# --- 2. Multinomial Naive Bayes ---
print("Training Multinomial Naive Bayes Model...")
t_start = time.time()
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)
t_train_nb = time.time() - t_start

# Evaluation
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)
nb_precision, nb_recall, nb_f1, _ = precision_recall_fscore_support(y_test, nb_preds, average='macro')

results['Multinomial Naive Bayes'] = {
    'Accuracy': nb_acc,
    'Precision': nb_precision,
    'Recall': nb_recall,
    'F1-Score': nb_f1,
    'Train Time (s)': t_train_nb
}
confusion_matrices['Multinomial Naive Bayes'] = confusion_matrix(y_test, nb_preds)

print(f"Naive Bayes trained in {t_train_nb:.2f} seconds.")
print("\nClassification Report:")
print(classification_report(y_test, nb_preds, target_names=['Negative', 'Neutral', 'Positive']))


### 8. Model Comparison and Evaluation Plots
Let's visualize and compare performance metrics and confusion matrices for both classifiers.



In [ ]:
# Convert results to dataframe
metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
print(metrics_df)

# Plot performance metrics comparison
melted_df = pd.melt(metrics_df, id_vars=['Model'], value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                    var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 7))
ax = sns.barplot(x='Metric', y='Score', hue='Model', data=melted_df, palette='muted')
plt.title('Model Performance Comparison (TF-IDF Features)', fontsize=15, fontweight='bold')
plt.ylabel('Score (0.0 to 1.0)')
plt.ylim(0, 1.1)

# Add values on top of bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.4f}',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.tight_layout()
plt.savefig(os.path.join("..", "Visualizations", "model_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Plot side-by-side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
classes = ['Negative (-1)', 'Neutral (0)', 'Positive (1)']

for idx, (model_name, cm) in enumerate(confusion_matrices.items()):
    # Calculate percentage cell values
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=classes, yticklabels=classes,
                annot_kws={"size": 13, "weight": "bold"})
    
    # Annotate percentages
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[idx].text(j + 0.5, i + 0.7, f"({cm_normalized[i, j]*100:.1f}%)",
                           ha='center', va='center', color='darkslategray', fontsize=9)
            
    axes[idx].set_title(f'{model_name} Confusion Matrix', fontsize=14, fontweight='bold', pad=10)
    axes[idx].set_xlabel('Predicted Sentiment')
    axes[idx].set_ylabel('Actual Sentiment')

plt.tight_layout()
plt.savefig(os.path.join("..", "Visualizations", "confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()


### 9. Key Findings & Internship Conclusion

#### Model Evaluation Summary
1. **Logistic Regression** achieved an overall accuracy of **89.04%** with a macro F1-score of **87.88%**. It showed high effectiveness at predicting neutral (97% recall) and positive reviews (92% precision).
2. **Multinomial Naive Bayes** achieved an accuracy of **72.92%** and macro F1-score of **70.15%**. It trained almost instantaneously (0.02s) but struggled to distinguish negative tweets effectively (48% recall).
3. **Logistic Regression** is the recommended model for deployment in this scenario due to its significantly higher predictive performance across all metrics.

#### Key Insights from Data
* Positive comments frequently mention: **modi, vote, great, support, clean, good, welcome**.
* Negative comments frequently focus on: **nonsense, drama, corruption, clean, state, fail**.
* The sentiment distribution is slightly imbalanced, with **Positive** being the largest class (approx. 44.4%), followed by **Neutral** (approx. 33.9%), and **Negative** (approx. 21.8%).

---
*End of Notebook. Submission-ready for Oasis Infobyte Data Analytics Internship.*

